In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__))
using Finch, BenchmarkTools

  Activating project at `~/Finch.jl/benchmark`


### Case 1: Dense shape lifting

In [2]:
A = Tensor(Dense(Element(0.0)), collect(1.0:5.0))
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)

=== Normal ===
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    for i_3 = 1:A_lvl_stop
        A_lvl_q = (1 - 1) * A_lvl_stop + i_3
        A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
        s_val = A_lvl_2_val_2 + s_val
    end
    s_data.val = s_val
    (s = s_data,)
end

=== Specialized ===
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    for i_3 = 1:5
        A_lvl_q = (1 - 1) * 5 + i_3
        A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
        s_val = A_lvl_2_val_2 + s_val
    end
    s_data.val = s_val
    (s = s_data,)
end


---
SparseList specialization fires when the fiber position is a compile-time `literal`.

### Case 2: Empty vector — entire loop eliminated

In [3]:
A = Tensor(SparseList(Element(0.0)), 10_000)
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)

=== Normal ===
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_ptr = A_lvl.ptr
    A_lvl_idx = A_lvl.idx
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    A_lvl_q = A_lvl_ptr[1]
    A_lvl_q_stop = A_lvl_ptr[1 + 1]
    if A_lvl_q < A_lvl_q_stop
        A_lvl_i1 = A_lvl_idx[A_lvl_q_stop - 1]
    else
        A_lvl_i1 = 0
    end
    phase_stop = min(A_lvl_i1, A_lvl_stop)
    if phase_stop >= 1
        if A_lvl_idx[A_lvl_q] < 1
            A_lvl_q = Finch.scansearch(A_lvl_idx, 1, A_lvl_q, A_lvl_q_stop - 1)
        end
        while true
            A_lvl_i = A_lvl_idx[A_lvl_q]
            if A_lvl_i < phase_stop
                A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
                s_val = A_lvl_2_val_2 + s_val
                A_lvl_q += 1
            else
                phase_stop_3 = min(phase_stop, A_lvl_i)
                if A_lvl_i == phase_stop_3
        

### Case 3: Singleton (nnz=1)

In [4]:
A = Tensor(SparseList(Element(0.0)), fsparse([5000], [3.14], (10_000,)))
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)

=== Normal ===
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_ptr = A_lvl.ptr
    A_lvl_idx = A_lvl.idx
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    A_lvl_q = A_lvl_ptr[1]
    A_lvl_q_stop = A_lvl_ptr[1 + 1]
    if A_lvl_q < A_lvl_q_stop
        A_lvl_i1 = A_lvl_idx[A_lvl_q_stop - 1]
    else
        A_lvl_i1 = 0
    end
    phase_stop = min(A_lvl_i1, A_lvl_stop)
    if phase_stop >= 1
        if A_lvl_idx[A_lvl_q] < 1
            A_lvl_q = Finch.scansearch(A_lvl_idx, 1, A_lvl_q, A_lvl_q_stop - 1)
        end
        while true
            A_lvl_i = A_lvl_idx[A_lvl_q]
            if A_lvl_i < phase_stop
                A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
                s_val = A_lvl_2_val_2 + s_val
                A_lvl_q += 1
            else
                phase_stop_3 = min(phase_stop, A_lvl_i)
                if A_lvl_i == phase_stop_3
        

### Case 4: Small nnz (≤4)

In [5]:
A = Tensor(SparseList(Element(0.0)), fsparse([10, 200, 5000], [1.0, 2.0, 3.0], (10_000,)))
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)

=== Normal ===


begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_ptr = A_lvl.ptr
    A_lvl_idx = A_lvl.idx
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    A_lvl_q = A_lvl_ptr[1]
    A_lvl_q_stop = A_lvl_ptr[1 + 1]
    if A_lvl_q < A_lvl_q_stop
        A_lvl_i1 = A_lvl_idx[A_lvl_q_stop - 1]
    else
        A_lvl_i1 = 0
    end
    phase_stop = min(A_lvl_i1, A_lvl_stop)
    if phase_stop >= 1
        if A_lvl_idx[A_lvl_q] < 1
            A_lvl_q = Finch.scansearch(A_lvl_idx, 1, A_lvl_q, A_lvl_q_stop - 1)
        end
        while true
            A_lvl_i = A_lvl_idx[A_lvl_q]
            if A_lvl_i < phase_stop
                A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
                s_val = A_lvl_2_val_2 + s_val
                A_lvl_q += 1
            else
                phase_stop_3 = min(phase_stop, A_lvl_i)
                if A_lvl_i == phase_stop_3
                    A_l

### Case 5: Contiguous block (above unroll threshold)

In [6]:
A = Tensor(SparseList(Element(0.0)), fsparse(collect(500:600), collect(1.0:101.0), (10_000,)))
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for i = _
        s[] += A[i]
    end
end)

=== Normal ===


begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.rhs.tns.bind.lvl
    A_lvl_ptr = A_lvl.ptr
    A_lvl_idx = A_lvl.idx
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_val = A_lvl_2.val
    s_val = 0.0
    A_lvl_q = A_lvl_ptr[1]
    A_lvl_q_stop = A_lvl_ptr[1 + 1]
    if A_lvl_q < A_lvl_q_stop
        A_lvl_i1 = A_lvl_idx[A_lvl_q_stop - 1]
    else
        A_lvl_i1 = 0
    end
    phase_stop = min(A_lvl_i1, A_lvl_stop)
    if phase_stop >= 1
        if A_lvl_idx[A_lvl_q] < 1
            A_lvl_q = Finch.scansearch(A_lvl_idx, 1, A_lvl_q, A_lvl_q_stop - 1)
        end
        while true
            A_lvl_i = A_lvl_idx[A_lvl_q]
            if A_lvl_i < phase_stop
                A_lvl_2_val_2 = A_lvl_2_val[A_lvl_q]
                s_val = A_lvl_2_val_2 + s_val
                A_lvl_q += 1
            else
                phase_stop_3 = min(phase_stop, A_lvl_i)
                if A_lvl_i == phase_stop_3
                    A_l

### Challenge 1: Only fires when fiber position is a compile-time literal

Specialization requires `pos` to be a `literal` node in the IR. This only holds for the outermost SparseList (pos=1) or when an outer level is unrolled (cascading literal positions to inner levels).

For the common `Dense(SparseList)` CSC format, the inner SparseList position depends on the Dense loop variable → runtime `value` → no specialization.

In [7]:
# Inner SparseList NOT specialized — only the outer Dense loop bound changes
A = Tensor(Dense(SparseList(Element(0.0))), fsparse([1], [1], [1.0], (3, 2)))
s = Scalar(0.0)

println("=== Normal ===")
println(@finch_code begin
    s .= 0.0
    for j = _, i = _
        s[] += A[i, j]
    end
end)
println("\n=== Specialized ===")
println(@finch_code specialize=true begin
    s .= 0.0
    for j = _, i = _
        s[] += A[i, j]
    end
end)

=== Normal ===
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.lvl
    A_lvl_stop = A_lvl.shape
    A_lvl_2 = A_lvl.lvl
    A_lvl_2_ptr = A_lvl_2.ptr
    A_lvl_2_idx = A_lvl_2.idx
    A_lvl_2_stop = A_lvl_2.shape
    A_lvl_3 = A_lvl_2.lvl
    A_lvl_3_val = A_lvl_3.val
    s_val = 0.0
    for j_3 = 1:A_lvl_stop
        A_lvl_q = (1 - 1) * A_lvl_stop + j_3
        A_lvl_2_q = A_lvl_2_ptr[A_lvl_q]
        A_lvl_2_q_stop = A_lvl_2_ptr[A_lvl_q + 1]
        if A_lvl_2_q < A_lvl_2_q_stop
            A_lvl_2_i1 = A_lvl_2_idx[A_lvl_2_q_stop - 1]
        else
            A_lvl_2_i1 = 0
        end
        phase_stop = min(A_lvl_2_i1, A_lvl_2_stop)
        if phase_stop >= 1
            if A_lvl_2_idx[A_lvl_2_q] < 1
                A_lvl_2_q = Finch.scansearch(A_lvl_2_idx, 1, A_lvl_2_q, A_lvl_2_q_stop - 1)
            end
            while true
                A_lvl_2_i = A_lvl_2_idx[A_lvl_2_q]
                if A_lvl_2_i < pha

Only visible difference: `A_lvl_stop` → `2` (the Dense shape). The inner Stepper loop body is identical.

### Challenge 2: Write protocols excluded

Write protocols (`extrude`, `defaultupdate`) assemble structure dynamically. ptr/idx are being created, not read. Specialization only applies to read protocols (`walk`, `gallop`, `follow`).